In [1]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

# Set device and random seed for reproducible results
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(42)

print(f"Using compute device: {device}")

Using compute device: cpu


MINI-TRANSFORMER MODEL (SESSIONS 8, 10, 11)

In [2]:
class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        norm_x = x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
        return self.weight * norm_x

In [3]:
class CausalSelfAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int, max_seq_len: int = 128):
        super().__init__()
        assert d_model % n_heads == 0, "d_model must be divisible by n_heads"
        self.d_model = d_model
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads

        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)

        mask = torch.triu(torch.ones(max_seq_len, max_seq_len), diagonal=1).bool()
        self.register_buffer("causal_mask", mask)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, T, C = x.shape
        q = self.q_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)

        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        scores = scores.masked_fill(self.causal_mask[:T, :T], float("-inf"))
        attn_weights = F.softmax(scores, dim=-1)

        out = (attn_weights @ v).transpose(1, 2).contiguous().view(B, T, C)
        return self.out_proj(out)


class SwiGLUFFN(nn.Module):
    def __init__(self, d_model: int, hidden_dim: int = None):
        super().__init__()
        if hidden_dim is None:
            hidden_dim = int(2 * (4 * d_model) / 3)
        self.w1 = nn.Linear(d_model, hidden_dim, bias=False)
        self.w2 = nn.Linear(hidden_dim, d_model, bias=False)
        self.w3 = nn.Linear(d_model, hidden_dim, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.w2(F.silu(self.w1(x)) * self.w3(x))

In [4]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model: int, n_heads: int):
        super().__init__()
        self.ln1 = RMSNorm(d_model)
        self.attn = CausalSelfAttention(d_model, n_heads)
        self.ln2 = RMSNorm(d_model)
        self.ffn = SwiGLUFFN(d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.attn(self.ln1(x))
        x = x + self.ffn(self.ln2(x))
        return x

In [5]:
class MiniGPT(nn.Module):
    def __init__(self, vocab_size: int = 256, d_model: int = 256, n_heads: int = 4, n_layers: int = 2):
        super().__init__()
        self.d_model = d_model
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(128, d_model)
        self.blocks = nn.ModuleList([TransformerBlock(d_model, n_heads) for _ in range(n_layers)])
        self.ln_f = RMSNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.tok_emb.weight = self.lm_head.weight

    def forward(self, idx: torch.Tensor, targets: torch.Tensor = None):
        B, T = idx.shape
        pos = torch.arange(0, T, device=idx.device).unsqueeze(0)
        x = self.tok_emb(idx) + self.pos_emb(pos)
        for block in self.blocks:
            x = block(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

In [6]:
# Generate dummy autoregressive sequence data for training tasks
def get_batch(vocab_size=256, batch_size=4, seq_len=32):
    x = torch.randint(0, vocab_size, (batch_size, seq_len), device=device)
    y = torch.randint(0, vocab_size, (batch_size, seq_len), device=device)
    return x, y

TASK 1: REPRODUCE ADAM BY HAND

In [7]:
print("\n" + "="*50)
print("TASK 1: Hand-written Adam vs PyTorch Adam Verification")
print("="*50)

# Setup initial weight and sequence of 5 test gradients
w_initial = 1.5
w_torch = torch.tensor([w_initial], requires_grad=True)
gradients = [0.1, -0.2, 0.05, 0.3, -0.1]

lr = 1e-3
beta1, beta2 = 0.9, 0.999
eps = 1e-8

optimizer_task1 = torch.optim.Adam([w_torch], lr=lr, betas=(beta1, beta2), eps=eps)

m_hand, v_hand = 0.0, 0.0
w_hand = w_initial

print(f"{'Step':<5} | {'m_hat':<10} | {'v_hat':<10} | {'Hand Weight':<12} | {'PyTorch Weight':<12} | {'Diff':<10}")
print("-" * 70)

for t, g in enumerate(gradients, 1):
    # --- PyTorch Step ---
    optimizer_task1.zero_grad()
    w_torch.grad = torch.tensor([g])
    optimizer_task1.step()

    # --- Manual Hand Step ---
    m_hand = beta1 * m_hand + (1 - beta1) * g
    v_hand = beta2 * v_hand + (1 - beta2) * (g ** 2)
    m_hat = m_hand / (1 - beta1 ** t)
    v_hat = v_hand / (1 - beta2 ** t)
    step = lr * m_hat / (math.sqrt(v_hat) + eps)
    w_hand -= step

    diff = abs(w_hand - w_torch.item())
    print(f"{t:<5} | {m_hat:<10.6f} | {v_hat:<10.6f} | {w_hand:<12.7f} | {w_torch.item():<12.7f} | {diff:<10.2e}")


TASK 1: Hand-written Adam vs PyTorch Adam Verification
Step  | m_hat      | v_hat      | Hand Weight  | PyTorch Weight | Diff      
----------------------------------------------------------------------
1     | 0.100000   | 0.010000   | 1.4990000    | 1.4990000    | 4.68e-08  
2     | -0.057895  | 0.025008   | 1.4993661    | 1.4993660    | 5.86e-08  
3     | -0.018081  | 0.017497   | 1.4995028    | 1.4995028    | 1.62e-08  
4     | 0.074411   | 0.035650   | 1.4991087    | 1.4991087    | 2.22e-08  
5     | 0.031821   | 0.030510   | 1.4989265    | 1.4989265    | 2.37e-09  


TASK 2: DISABLE BIAS CORRECTION & PLOT FIRST 20 STEPS

In [8]:
print("\n" + "="*50)
print("TASK 2: Standard Adam vs Uncorrected Adam (First 20 Steps)")
print("="*50)

# Create 100 fixed gradients to track convergence horizon
torch.manual_seed(100)
test_grads = torch.randn(500).tolist()

# Run Standard Adam (with bias correction)
m_corr, v_corr = 0.0, 0.0
w_corr = 1.0
trajectory_corr = []

for t, g in enumerate(test_grads, 1):
    m_corr = beta1 * m_corr + (1 - beta1) * g
    v_corr = beta2 * v_corr + (1 - beta2) * (g ** 2)
    m_hat = m_corr / (1 - beta1 ** t)
    v_hat = v_corr / (1 - beta2 ** t)
    w_corr -= lr * m_hat / (math.sqrt(v_hat) + eps)
    trajectory_corr.append(w_corr)

# Run Uncorrected Adam (bias correction disabled)
m_uncorr, v_uncorr = 0.0, 0.0
w_uncorr = 1.0
trajectory_uncorr = []

for t, g in enumerate(test_grads, 1):
    m_uncorr = beta1 * m_uncorr + (1 - beta1) * g
    v_uncorr = beta2 * v_uncorr + (1 - beta2) * (g ** 2)
    # Direct uncorrected updates
    w_uncorr -= lr * m_uncorr / (math.sqrt(v_uncorr) + eps)
    trajectory_uncorr.append(w_uncorr)

# Find step where difference becomes negligible (< 1%)
threshold_step = None
for idx, (c, u) in enumerate(zip(trajectory_corr, trajectory_uncorr), 1):
    rel_diff = abs(c - u) / abs(c)
    if rel_diff < 0.01 and threshold_step is None and idx > 10:
        threshold_step = idx

print(f"Step at which bias correction difference drops below 1%: Step {threshold_step}")

# Plot Task 2 (First 20 Steps)
plt.figure(figsize=(8, 4))
plt.plot(range(1, 21), trajectory_corr[:20], label="Standard Adam (Bias Corrected)", marker="o")
plt.plot(range(1, 21), trajectory_uncorr[:20], label="Uncorrected Adam (Bias Disabled)", marker="s", linestyle="--")
plt.xlabel("Step")
plt.ylabel("Weight Value")
plt.title("Task 2: Effect of Bias Correction in First 20 Steps")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig("task2_bias_correction.png")
plt.close()
print("Saved Task 2 plot to 'task2_bias_correction.png'")


TASK 2: Standard Adam vs Uncorrected Adam (First 20 Steps)
Step at which bias correction difference drops below 1%: Step 23
Saved Task 2 plot to 'task2_bias_correction.png'


TASK 3: LOG UPDATE-TO-WEIGHT RATIO & WARMUP

In [9]:
print("\n" + "="*50)
print("TASK 3: Layer-wise Update-to-Weight Ratio & Warmup Detection")
print("="*50)

model_task3 = MiniGPT(d_model=256).to(device)
optimizer_task3 = torch.optim.AdamW(model_task3.parameters(), lr=1e-3)
warmup_steps = 30
total_steps_task3 = 100

ratio_history = {name: [] for name, _ in model_task3.named_parameters()}

for step in range(1, total_steps_task3 + 1):
    # Linear Warmup Schedule
    current_lr = 1e-3 * min(1.0, step / warmup_steps)
    for param_group in optimizer_task3.param_groups:
        param_group['lr'] = current_lr

    x, y = get_batch()
    optimizer_task3.zero_grad()
    _, loss = model_task3(x, y)
    loss.backward()
    optimizer_task3.step()

    # Log Update-to-Weight Ratio for each parameter
    with torch.no_grad():
        for name, param in model_task3.named_parameters():
            if param.requires_grad and param.grad is not None:
                state = optimizer_task3.state[param]
                if "exp_avg" in state:
                    m = state["exp_avg"]
                    v = state["exp_avg_sq"]
                    st = state["step"]
                    m_hat = m / (1 - beta1 ** st)
                    v_hat = v / (1 - beta2 ** st)
                    update = current_lr * m_hat / (torch.sqrt(v_hat) + eps)
                    ratio = (torch.norm(update) / torch.norm(param)).item()
                    ratio_history[name].append(ratio)

# Identify stabilization step post-warmup for a key layer
sample_layer = "blocks.0.attn.q_proj.weight"
sample_ratios = ratio_history[sample_layer]
peak_step = sample_ratios.index(max(sample_ratios)) + 1
print(f"Sample Layer: '{sample_layer}'")
print(f"Configured Warmup End: Step {warmup_steps}")
print(f"Step at which Update-to-Weight Ratio stops increasing: Step {peak_step}")


TASK 3: Layer-wise Update-to-Weight Ratio & Warmup Detection
Sample Layer: 'blocks.0.attn.q_proj.weight'
Configured Warmup End: Step 30
Step at which Update-to-Weight Ratio stops increasing: Step 47


TASK 4: COSINE VS WSD SCHEDULE (300 STEPS, STOP AT STEP 200)

In [10]:
print("\n" + "="*50)
print("TASK 4: Cosine vs WSD Training Comparison (Stopped at Step 200)")
print("="*50)

def run_schedule(schedule_type):
    torch.manual_seed(42)
    model = MiniGPT(d_model=256).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
    losses = []

    total_steps = 300
    warmup = 30
    decay_start = 240  # WSD decays last 20% (steps 240-300)

    for step in range(1, 201):  # Stop evaluation at step 200
        if schedule_type == "cosine":
            if step <= warmup:
                lr_step = 1e-3 * (step / warmup)
            else:
                progress = (step - warmup) / (total_steps - warmup)
                lr_step = 1e-3 * 0.5 * (1.0 + math.cos(math.pi * progress))
        elif schedule_type == "wsd":
            if step <= warmup:
                lr_step = 1e-3 * (step / warmup)
            elif step <= decay_start:
                lr_step = 1e-3  # Stable phase
            else:
                progress = (step - decay_start) / (total_steps - decay_start)
                lr_step = 1e-3 * 0.5 * (1.0 + math.cos(math.pi * progress))

        for param_group in optimizer.param_groups:
            param_group['lr'] = lr_step

        x, y = get_batch()
        optimizer.zero_grad()
        _, loss = model(x, y)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())

    return losses

losses_cosine = run_schedule("cosine")
losses_wsd = run_schedule("wsd")

print(f"Cosine Loss at Step 200: {losses_cosine[-1]:.4f}")
print(f"WSD Loss at Step 200:    {losses_wsd[-1]:.4f}")
print("\nRecommendation:")
print("- At Step 200, Cosine reports lower loss because it has already aggressively decayed its LR.")
print("- However, we should KEEP the WSD model/checkpoint because its LR remained in the high stable phase.")
print("  WSD retains optimization mobility and can be decayed flexibly at step 200 or extended to higher token budgets.")


TASK 4: Cosine vs WSD Training Comparison (Stopped at Step 200)
Cosine Loss at Step 200: 5.5545
WSD Loss at Step 200:    5.6197

Recommendation:
- At Step 200, Cosine reports lower loss because it has already aggressively decayed its LR.
- However, we should KEEP the WSD model/checkpoint because its LR remained in the high stable phase.
  WSD retains optimization mobility and can be decayed flexibly at step 200 or extended to higher token budgets.


TASK 5: WIDTH SWEEP (256, 512, 1024) & PREDICT FOR WIDTH 4096

In [11]:
print("\n" + "="*50)
print("TASK 5: Learning Rate Sweep Across Model Widths & Extrapolation")
print("="*50)

widths = [256, 512, 1024]
lr_candidates = [1e-4, 3e-4, 1e-3, 3e-3, 1e-2]
optimal_lrs = {}

plt.figure(figsize=(8, 5))

for w in widths:
    n_heads = max(1, w // 64)
    width_losses = []

    for test_lr in lr_candidates:
        torch.manual_seed(42)
        model = MiniGPT(d_model=w, n_heads=n_heads).to(device)
        optimizer = torch.optim.AdamW(model.parameters(), lr=test_lr)

        # Train for 50 steps per candidate LR
        for _ in range(50):
            x, y = get_batch()
            optimizer.zero_grad()
            _, loss = model(x, y)
            loss.backward()
            optimizer.step()

        width_losses.append(loss.item())

    best_idx = width_losses.index(min(width_losses))
    best_lr = lr_candidates[best_idx]
    optimal_lrs[w] = best_lr

    plt.plot(lr_candidates, width_losses, marker="o", label=f"Width {w} (Min @ {best_lr})")
    print(f"Width {w:<4} | Minima Loss @ LR = {best_lr}")

# Predict LR for Width 4096 using Standard PyTorch Parameterization (SP) scaling (~ d^-0.5 to d^-1)
# Under SP, optimal learning rate scales inversely with width: lr_4096 = lr_1024 * sqrt(1024 / 4096)
predicted_lr_4096 = optimal_lrs[1024] * math.sqrt(1024 / 4096)

print("\n--- Extrapolation for Width 4,096 ---")
print(f"Predicted Optimal LR for Width 4096: {predicted_lr_4096:.5f}")
print("Confidence Level: Moderate (7/10)")
print("Justification: Standard PyTorch parameterization (SP) causes optimal learning rate to scale inversely")
print("with width (lr ∝ 1/√d) to preserve feature stability. Confidence is bounded by short step counts and fixed batch size.")

plt.xscale("log")
plt.xlabel("Learning Rate")
plt.ylabel("Loss after 50 steps")
plt.title("Task 5: LR Sweep across Model Widths")
plt.legend()
plt.grid(True, which="both", ls="--")
plt.tight_layout()
plt.savefig("task5_width_sweep.png")
plt.close()
print("Saved Task 5 plot to 'task5_width_sweep.png'")


TASK 5: Learning Rate Sweep Across Model Widths & Extrapolation
Width 256  | Minima Loss @ LR = 0.0003
Width 512  | Minima Loss @ LR = 0.001
Width 1024 | Minima Loss @ LR = 0.0003

--- Extrapolation for Width 4,096 ---
Predicted Optimal LR for Width 4096: 0.00015
Confidence Level: Moderate (7/10)
Justification: Standard PyTorch parameterization (SP) causes optimal learning rate to scale inversely
with width (lr ∝ 1/√d) to preserve feature stability. Confidence is bounded by short step counts and fixed batch size.
Saved Task 5 plot to 'task5_width_sweep.png'
